In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers datasets accelerate

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("GPT-2 model loaded successfully! ✅")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 model loaded successfully! ✅


In [4]:
training_text = """
Artificial intelligence is transforming the way people work and learn.
Machine learning allows computers to learn patterns from data.
Generative AI can create text, images, code, and other forms of content.
Python is one of the most popular programming languages for artificial intelligence.
Natural language processing helps computers understand and generate human language.
Deep learning uses neural networks to solve complex problems.
Large language models can generate meaningful text based on a given prompt.
GPT-2 is a transformer-based language model that can generate human-like text.
Fine-tuning allows a pre-trained model to learn from a specific custom dataset.
AI applications are becoming increasingly useful in education, healthcare, and business.
Learning artificial intelligence requires practice, experimentation, and continuous learning.
Data is an important component of modern machine learning systems.
A well-designed dataset can help a model learn the style and structure of the training text.
Text generation models predict the next token based on the context provided to them.
Generative artificial intelligence is an important area of modern technology.
"""

with open("custom_dataset.txt", "w", encoding="utf-8") as file:
    file.write(training_text)

print("Custom dataset created successfully! ✅")

Custom dataset created successfully! ✅


In [5]:
from datasets import load_dataset

# Load our custom text dataset
dataset = load_dataset(
    "text",
    data_files={"train": "custom_dataset.txt"}
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 16
    })
})


In [6]:
# GPT-2 does not have a padding token by default,
# so we use the EOS token as the padding token.
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Dataset tokenized successfully! ✅")

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Dataset tokenized successfully! ✅


In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=1,
    logging_steps=10,
    learning_rate=5e-5,
    report_to="none"
)

print("Training configuration created successfully! ✅")

Training configuration created successfully! ✅


In [11]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    processing_class=tokenizer
)

print("Trainer created successfully! ✅")

Trainer created successfully! ✅


In [12]:
from transformers import DataCollatorForLanguageModeling, Trainer

# Create data collator for GPT-2 causal language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully! ✅")

Trainer created successfully! ✅


In [13]:
# Start GPT-2 fine-tuning
print("Starting GPT-2 fine-tuning... 🚀")

trainer.train()

print("GPT-2 fine-tuning completed successfully! ✅")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting GPT-2 fine-tuning... 🚀


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.275557
20,2.232704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GPT-2 fine-tuning completed successfully! ✅


In [14]:
import torch

# Give GPT-2 a prompt
prompt = "Artificial intelligence"

# Convert prompt into tokens
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate text
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

# Convert generated tokens back to text
generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated Text:\n")
print(generated_text)

Generated Text:

Artificial intelligence is a key technology that can make a person's life more fulfilling.

In a paper published online on the Internet Research, researchers from the University of California, Berkeley and the University of California, Berkeley, have developed a system for learning and understanding human language. The system is based on a neural network, which can recognize human language.

The system uses the natural language learning to understand and


In [15]:
# Prompts for testing the fine-tuned GPT-2 model
prompts = [
    "Artificial intelligence",
    "Machine learning",
    "Generative AI"
]

results = []

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.8,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    results.append(f"PROMPT: {prompt}\n")
    results.append(f"GENERATED TEXT:\n{generated_text}\n")
    results.append("-" * 80 + "\n")

# Save results to a text file
with open("generation_results.txt", "w", encoding="utf-8") as file:
    file.writelines(results)

print("Generation results saved successfully! ✅")

Generation results saved successfully! ✅


In [16]:
!cat generation_results.txt

PROMPT: Artificial intelligence
GENERATED TEXT:
Artificial intelligence systems are used to learn and learn from human behavior. In this paper, we describe the neural network of AI systems. The neural networks that can learn and learn from human behavior are the foundations for intelligent systems.

What is an AI system?

An AI system is a system that learns from human experience. Artificial intelligence systems can learn from human behavior.

An AI is a system
--------------------------------------------------------------------------------
PROMPT: Machine learning
GENERATED TEXT:
Machine learning is an area of technology and a broad range of applications. A number of these applications, from machine learning to neural networks, are being used for all of today's problems.
In this article, we focus on a broad range of applications and the applications of machine learning. The goal of this article is to demonstrate the potential of machine learning and related technologies to address a w

In [17]:
# Save the fine-tuned GPT-2 model
trainer.save_model("./gpt2-finetuned-final")

# Save the tokenizer
tokenizer.save_pretrained("./gpt2-finetuned-final")

print("Fine-tuned GPT-2 model saved successfully! ✅")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned GPT-2 model saved successfully! ✅


In [18]:
import os

print("Saved model files:")
for file in os.listdir("gpt2-finetuned-final"):
    print(file)

Saved model files:
training_args.bin
model.safetensors
config.json
tokenizer_config.json
tokenizer.json
generation_config.json
